In [1]:
import os, sys, importlib
from pypdf import PdfReader

# test

## read pdf

In [ ]:
import pdfplumber
def words_per_page(path_pdf, start_page=10):
    
    with pdfplumber.open(path_pdf) as pdf:
        pages = pdf.pages[start_page:start_page+1]
        for i, page in enumerate(pages):
            words = page.extract_words()
    return words


In [ ]:
# regroupe les textes selon leur coordonnée "bottom"

from collections import defaultdict
def group_words_into_lines(words, y_tolerance=2):
    lines = defaultdict(list)
    for w in words:
        # 用四舍五入避免轻微浮动
        line_key = round(w["bottom"] / y_tolerance) * y_tolerance
        lines[line_key].append(w)

    # 按从上到下排序
    sorted_lines = []
    for key in sorted(lines.keys()):
        # 每行内部按 x0 排序（从左到右）
        line_words = sorted(lines[key], key=lambda x: x["x0"])
        line_text = " ".join(w["text"] for w in line_words)
        sorted_lwordsines.append(line_text.strip())
    return sorted_lines


In [6]:
# test:

path_pdf="../data\catalogue_of_romance_vol1.pdf"
words=words_per_page(path_pdf, start_page=9)
lines = group_words_into_lines(words,y_tolerance=4)
for line in lines:
    print(line)

TABLE OF CONTENTS.
Classical Bomamces.
Cycle of Troy: — pagb
Heroica, by FlaviuB Philostratus 1
Iliaca, by Joannes Tzetzes .. .. 2
Dictys Cretensis .. .. .. .. .. .. .. 9
Dares Phrygiiu .. .. .. .. .. .. .. 12
Latin Poems, by Simon Chdvre d*Or, etc. 27
Boman de Troie, by Benolt de Sainte-More .. .. 35
Historia Trojana, by Goido delle Colonne . .. 40
Tr6jamannaSaga .. .. .. .. .. .. .. .. 62
Blmur af Tr6jumonnum, by J6n J6ns8on 63
Becueil, by Raoul Le Fdvre .. .. .. .. .. 64
Latin History of Troy 66
Filostrato, by Boccaccio .. .. .. .. 67
Troilus and Ghryseyde, by Chaucer 70
Troy Book, by Lydgate 76
Boman d'Eneas .. .. .. .. .. 82
Bomanoe of Troy .. .. .. •• .. .. 84
Romance of Thebes, by Lydgate .. .. .. .. 87
Bomanoe of Jason, from Baoul Le Fdvre (Dutch) 92
Cycle of Alexander :—
Alezandreis, by Gatiltier de Ch&tiUon .. 94
Alexanders Saga, by Brandr J6nsson .. 104
Alexander, abridgisd from Julius Valerius .. .. 106
Alexander, or Historia de Preliis .. .. 120
Marvels of India, etc .. ..

In [8]:
START_PAGE=9
END_PAGE=13

all_lines=[]
for start_page in range(START_PAGE, END_PAGE):
    print(f"page {start_page}".center(100,"-"))
    words=words_per_page(path_pdf, start_page)
    lines = group_words_into_lines(words)
    all_lines.extend(lines)
    for line in lines:
        print(line)

-----------------------------------------------page 9-----------------------------------------------
TABLE OF CONTENTS.
Classical Bomamces.
Cycle of Troy: — pagb
Heroica, by FlaviuB Philostratus 1
Iliaca, by Joannes Tzetzes .. .. 2
Dictys Cretensis .. .. .. .. .. .. .. 9
Dares Phrygiiu .. .. .. .. .. .. .. 12
Latin Poems, by Simon Chdvre d*Or, etc. 27
Boman de Troie, by Benolt de Sainte-More .. .. 35
Historia Trojana, by Goido delle Colonne . .. 40
Tr6jamannaSaga .. .. .. .. .. .. .. .. 62
Blmur af Tr6jumonnum, by J6n J6ns8on 63
Becueil, by Raoul Le Fdvre .. .. .. .. .. 64
Latin History of Troy 66
Filostrato, by Boccaccio .. .. .. .. 67
Troilus and Ghryseyde, by Chaucer 70
Troy Book, by Lydgate 76
Boman d'Eneas .. .. .. .. .. 82
Bomanoe of Troy .. .. .. •• .. .. 84
Romance of Thebes, by Lydgate .. .. .. .. 87
Bomanoe of Jason, from Baoul Le Fdvre (Dutch) 92
Cycle of Alexander :—
Alezandreis, by Gatiltier de Ch&tiUon .. 94
Alexanders Saga, by Brandr J6nsson .. 104
Alexander, abridgisd f

## clean lines:


In [9]:
# No TABLE OF CONTENTS.
# NO Cycle of xxx.
# NO CONTENTS.
# correct bad themes ocr

lines = all_lines.copy()
lines = [l.strip() for l in lines if l.strip() and not l.startswith("TABLE") and not l.lower().startswith("cycle")]
import re

def is_contents_line(line):
    clean = re.sub(r'[^A-Za-z]', '', line).lower()
    return 'contents' in clean
lines = [l for l in lines if not is_contents_line(l)]


ocr_to_clean = {
    'Classical Bomamces.': 'Classical Romances',
    'BSTTIBH AND EnQLISH TRADITIONS.': 'British and English Traditions',
    'Fbench Tbaditions.': 'French Traditions',
    'MlBOKLLAlISOUB R0MAMCB8.': 'Miscellaneous Romance',
    'Allbgobioal akd DiDAcno Romakces.': 'Allegorical and Didactic Romance',
    'Appendix.': 'Appendix'
}
lines=[ocr_to_clean[l] if l in ocr_to_clean.keys() else l for l in lines]
lines

['Classical Romances',
 'Heroica, by FlaviuB Philostratus 1',
 'Iliaca, by Joannes Tzetzes .. .. 2',
 'Dictys Cretensis .. .. .. .. .. .. .. 9',
 'Dares Phrygiiu .. .. .. .. .. .. .. 12',
 'Latin Poems, by Simon Chdvre d*Or, etc. 27',
 'Boman de Troie, by Benolt de Sainte-More .. .. 35',
 'Historia Trojana, by Goido delle Colonne . .. 40',
 'Tr6jamannaSaga .. .. .. .. .. .. .. .. 62',
 'Blmur af Tr6jumonnum, by J6n J6ns8on 63',
 'Becueil, by Raoul Le Fdvre .. .. .. .. .. 64',
 'Latin History of Troy 66',
 'Filostrato, by Boccaccio .. .. .. .. 67',
 'Troilus and Ghryseyde, by Chaucer 70',
 'Troy Book, by Lydgate 76',
 "Boman d'Eneas .. .. .. .. .. 82",
 'Bomanoe of Troy .. .. .. •• .. .. 84',
 'Romance of Thebes, by Lydgate .. .. .. .. 87',
 'Bomanoe of Jason, from Baoul Le Fdvre (Dutch) 92',
 'Alezandreis, by Gatiltier de Ch&tiUon .. 94',
 'Alexanders Saga, by Brandr J6nsson .. 104',
 'Alexander, abridgisd from Julius Valerius .. .. 106',
 'Alexander, or Historia de Preliis .. .. 120',

## organise data:

In [ ]:
import re
import json

def parse_toc(lines, themes):
    """
    input：
        lines: OCR 文本行列表
        themes: 标准主题列表
    output：
        toc_data: {theme: [{"title": ..., "page": ...}, ...], ...}
    """
    toc_data = {}
    current_theme = None

    for line in lines:
        line_strip = line.strip()
        if not line_strip:
            continue

        # 1️⃣ si thème
        matched = False
        for theme in themes:
            if line_strip.lower().startswith(theme.lower()[:6]):  # OCR 容错前6字符
                current_theme = theme
                toc_data[current_theme] = []
                matched = True
                break
        if matched:
            continue

        # 2️⃣ traiter la ligne
        if current_theme:
            text = line_strip

            # a) 去掉连续点填充（dot leader）
            text = re.sub(r'\.{1,}', '', text) #去掉连续n个及以上的点

            # b) 去掉其他 OCR 噪音字符，只保留字母、数字、空格和常用标点
            text = re.sub(r"[^A-Za-z0-9 ,.'()\-\u00C0-\u017F]", "", text)

            # c) 合并多空格
            text_clean = re.sub(r'\s+', ' ', text).strip()

            # d) 提取末尾页码
            m = re.search(r'(\d+)$', text_clean)
            if m:
                page = m.group(1)
                title = text_clean[:m.start()].strip()
            else:
                page = None
                title = text_clean

            if title:  # 避免空行加入
                toc_data[current_theme].append({
                    "title": title,
                    "page": page
                })

    return toc_data

# -------------------
themes=ocr_to_clean.values()
print(themes)
toc = parse_toc(lines, themes)
print(json.dumps(toc, indent=2, ensure_ascii=False))

dict_values(['Classical Romances', 'British and English Traditions', 'French Traditions', 'Miscellaneous Romance', 'Allegorical and Didactic Romance', 'Appendix'])
{
  "Classical Romances": [
    {
      "title": "Heroica, by FlaviuB Philostratus",
      "page": "1"
    },
    {
      "title": "Iliaca, by Joannes Tzetzes",
      "page": "2"
    },
    {
      "title": "Dictys Cretensis",
      "page": "9"
    },
    {
      "title": "Dares Phrygiiu",
      "page": "12"
    },
    {
      "title": "Latin Poems, by Simon Chdvre dOr, etc",
      "page": "27"
    },
    {
      "title": "Boman de Troie, by Benolt de Sainte-More",
      "page": "35"
    },
    {
      "title": "Historia Trojana, by Goido delle Colonne",
      "page": "40"
    },
    {
      "title": "Tr6jamannaSaga",
      "page": "62"
    },
    {
      "title": "Blmur af Tr6jumonnum, by J6n J6ns8on",
      "page": "63"
    },
    {
      "title": "Becueil, by Raoul Le Fdvre",
      "page": "64"
    },
    {
      "title":

In [46]:
path_results="../results/theme_table.json"
import json
with open(path_results, "w",encoding="utf-8") as f :
    json.dump(toc, f, indent=2, ensure_ascii=False)